*UID*:

Christopher Perez Lebron: 120405389

Andre Pham - 118830360

Ian Fuller: 117923346

Shasank Patel: 118976501

# **CMSC426 Project 1: Color Segmentation using GMM**

# Introduction

Have you ever played with these adorable Nao robots? Click [here](http://www.youtube.com/watch?feature=player_embedded&v=Gy_wbhQxd_0) to watch a cool demo.

Nao robots are star players in RoboCup, an annual autonomous robot soccer competitions. Would you like to help us in Nao’s soccer training? We need to train Nao to detect a soccer ball and estimate the depth of the ball to know how far to kick.

Nao’s training has two phases:

- Color Segmentation using Gaussian Mixture Model (GMM)
- Ball Distance Estimation

<a name='problem'></a>
# What you need to do

To make logistics easier, we have collected camera data from Nao robot on behalf of you and saved the data in the form of color images. Click [here](https://drive.google.com/file/d/17XiM86JqHqko4JC00-E4w4sPKnzh2iMz/view?usp=sharing) to download, or **run the following code block to download the training image folder to the file directory of the notebook**. The image names represent the depth of the ball from Nao robot in centimeters. -We will release the test dataset 48 hours before the deadline. **Test images are available [here](https://drive.google.com/file/d/17tNn3YIVR-8kqoBgJNK58YY4UBnQmm4q/view?usp=sharing) to download.**

In [ ]:
# Download training images from Google Drive
import gdown

gdown.download_folder(
    id="18Mx2Xc9UNFZYajYu9vfmRFlFCcna5I0J", quiet=True, use_cookies=False
)

In [ ]:
# Download testing images from Google Drive
gdown.download_folder(
    id="1Yl4_5O_ZEkz_KJVs0_vS5TrZUqMYkwr4", quiet=True, use_cookies=False
)

In [ ]:
# Check whether the training images were successfully imported
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

train_image = mpimg.imread("/content/train_images/106.jpg")
plt.imshow(train_image)
plt.axis("off")
plt.show()

In [ ]:
# # import cv2

# # #trying to find threshold w/ code
# image = cv2.imread("/content/train_images/106.jpg") #NOTE: cv2 stores rgb in oposite order (they store it as BRG)


# # image = cv2.imread("/content/ball.png")
# image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
# # print(len(image))
# # print(len(image[0]))

# # maxColor = [-1, -1, -1]
# # minColor = [300, 300, 300]

# # totalColor = [0,0,0]

# # count = 0

# # for row in image[1:-1]:
# #   for pixel in row[1:-1]:
# #     if not np.array_equal(pixel,  np.array([0, 0, 0])):

# #       for i in range(0, 3):
# #         if pixel[i] > maxColor[i]:
# #           maxColor[i] = pixel[i]
# #         elif pixel[i] < minColor[i]:
# #           minColor[i] = pixel[i]

# #         totalColor[0] += pixel[0]
# #         totalColor[1] += pixel[1]
# #         totalColor[2] += pixel[2]

# #         count += 1

# # avgColor = [0, 0, 0]
# # avgColor[0] = totalColor[0]/count
# # avgColor[1] = totalColor[1]/count
# # avgColor[2] = totalColor[2]/count


# # print(maxColor)
# # print(minColor)
# # print(avgColor)

# plt.imshow(cv2.cvtColor(image, cv2.COLOR_HSV2RGB)) #convert back to rgb before displaying w/ matlab
# plt.axis("off")
# plt.show()

## Problem Statement

1. Write Python code to cluster the orange ball using [Single Gaussian](https://cmsc426.github.io/colorseg/#gaussian) [30 points]

In [ ]:
# TODO: Import all python packages you need
import cv2
import os
import glob

# TODO: Read in training images
path = "/content/train_images"
files = "*.jpg"

file_path = glob.glob(os.path.join(path, files))
print(file_path)
total = []

# TODO: Iterate over training images to extract orange pixels using masks
for x in file_path:
    image = cv2.imread(x)  # read image x
    image_array = np.array(image)  # convert the image into an np array
    # lower = np.array([10, 100, 20]) #a BGR lowerbound, this is a dark green
    # upper = np.array([25, 255, 255]) #a BGR upperbound, this is a bright yellow

    # mostly just the ball (ONLY WORKS WHEN WE ASSUME IMAGE IS RGB, BUT IT
    # ACTUALLY COMES IN AS BGR)
    lower = np.array([110, 100, 100])
    upper = np.array([120, 255, 255])

    # trying to improve on last one

    hsv = cv2.cvtColor(
        image_array, cv2.COLOR_RGB2HSV
    )  # create new copy of image w/ HSV color space
    mask = cv2.inRange(
        hsv, lower, upper
    )  # grab the subset of the image who's color lies within the given bound
    # Question about previous line: does mask just tell you which pixels fall in the specified range?
    # Answer - The mask will use the lower and upper bounds given by the array of color values and matches pixels based on those ranges.

    # write original image to memory
    cv2.imwrite("x.jpg", image)
    # write the mask to memeory
    cv2.imwrite("mask.jpg", mask)

    # write hsv
    cv2.imwrite("hsv.jpg", cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))

    # write image_masked.jpg
    image_masked = cv2.bitwise_and(image, image, mask=mask)
    cv2.imwrite("image_masked.jpg", image_masked)

    orange_total = hsv[mask > 0]
    total.extend(orange_total)


total = np.array(total)
print(total)

# TODO: Compute mean and covariance using MLE(Maximum Likelihood Estimation)
mean = np.mean(total, axis=0)
print("mean: ")
print(mean)
covariance = np.cov(total.T)
# covariance = np.cov(total)

print(covariance)
# TODO: Compute PDF(Probability Density Function) of single gaussian model


def pdf(x, mean, covariance):
    diff = (x - mean).reshape(3, 1)  # (X - u) part of the equatation

    # FOR DEBUGGING
    # left = np.dot(x_m, np.linalg.inv(covariance))
    # print("LEFT: ")
    # print(left)

    # right = x_m.T
    # print("RIGHT: ")
    # print(right)
    # FOR DEBUGGING

    # exp_arg = -0.5 * np.dot(np.dot(x_m.T, np.linalg.inv(covariance)), x_m)
    # print(exp_arg[0][0])
    # exp = np.exp(np.clip(exp_arg[0][0], -700, 700))
    # this represents the exp part of the equation.
    # x_m.T represents the transpose of (x-u)
    # np.linalg.inv(covariance) represents the inverse of the covariance matrix.
    # All multiplied by -1/2

    # determinate is maxed b/c determinate can experience underflow making it default to 0 & breaking the sqrt function
    # pdf = exp / np.sqrt((2 * np.pi)**3 * max(np.linalg.det(covariance), 10**-300))

    # Putting it all together, the exp portion of the equation divided by the pdf
    # np.sqrt(2 * np.pi)** 3 *, represents sqrt(2pi)^3 part of the equation
    # np.linalg.det(covariance) represents |Sigma| which is the determiant of the cov. matrix

    # print(pdf)

    # TA HINT to fix singular matrix:
    # alpha = .01

    exponent = -0.5 * (diff.transpose() @ np.linalg.inv(covariance) @ diff)
    # exponent = exponent*alpha

    # print(f"difference: {diff}")
    # print(f"covariance: {covariance}")
    # print(f"covariance_inverse: {np.linalg.inv(covariance)}")
    # print(f"exp_arg: {exponent}")
    # exponent = -np.log(-exponent)
    numerator = (np.exp(exponent)).item()
    denominator = np.sqrt(((2 * np.pi) ** 3) * abs(np.linalg.det(covariance)))

    pdf = numerator / denominator

    # did something wrong to cause singular matrix (something in E-M algorithm)

    # the errors we saw underflow, overflow, singular matrix, etc is b/c we did expectation maximization algorithm wrong

    # pay special attention to anything going into np.linalg.det(cov)
    # so that means pay special attention to covariance matrix

    return pdf


# TODO: Set parameters (threshold, prior)
threshhold = 0.00000000000001


# P(Orange)
prior = 0.000000001


# TODO: Send test images into algorithm to detect orange ball
test_path = "/content/test_images"
test_files = "*.jpg"
test_file_path = glob.glob(os.path.join(test_path, test_files))

outputs = []

i = 0
for x in test_file_path:
    image = cv2.imread(x)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    originalImage = image.copy()

    for row in image:
        for pixel in row:

            # pass pixel into PDF to get P(x|Orange)
            likelihood = pdf(pixel, mean, covariance)

            # assumption: P(Orange|x) is proportional to P(x|Orange)*P(Orange)
            posterior = likelihood * prior

            if posterior >= threshhold:
                pixel[0] = 255
                pixel[1] = 255
                pixel[2] = 255
            else:
                pixel[0] = 0
                pixel[1] = 0
                pixel[2] = 0

    outputs.append((originalImage, image))
    # cv2.imwrite(f"singleGaussianOutput_{x.replace('/', '_')}.jpg", image)
    # print(x)
    # i += 1

    # plt.imshow(image)
    # plt.axis("off")
    # plt.show()

In [ ]:
# Show you result here

# TODO: change code to show original and output next to each other
# for image in outputs:
#   plt.imshow(image)
#   plt.axis("off")
#   plt.show()


for original, result in outputs:
    fig, axes = plt.subplots(1, 2, figsize=(24, 8))

    original_rbg = cv2.cvtColor(original, cv2.COLOR_HSV2BGR)
    # result_rbg = cv2.cvtColor(result, cv2.COLOR_HSV2RGB)
    # axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_HSV2RGB))
    axes[0].imshow(original_rbg)
    axes[0].set_title("Original")

    axes[1].imshow(result)
    axes[1].set_title("Model Output")

    plt.tight_layout()
    plt.show()

2. Write Python code to cluster the orange ball using [Gaussian Mixture Model](https://cmsc426.github.io/colorseg/#gmm) [40 points] and estimate the [distance](https://cmsc426.github.io/colorseg/#distest) to the ball [20 points]. Also, plot all the GMM ellipsoids [10 points].

You are NOT allowed to use any built-in Python package(s) like *sklearn.mixture.GaussianMixture* for GMM. To help you with code implementation, we have given the pseudocode :-)

In [ ]:
# print(np.random.rand(1) > np.random.rand(1))
# print(type(np.random.rand(1)))

# k = 3
# covariances = (np.random.rand(k*9) * 1000).reshape((-1, 3, 3))
# print(covariances)
# print(covariances[0])


# image = cv2.imread("/content/train_images/106.jpg")
# images = [image, image, image]
# pixels=image.reshape(-1, 3)

# for i, image in enumerate(images):
#   print(i)
#   print(image)

# images[:][3]

# exp / np.sqrt((2 * np.pi)**3 * np.linalg.det(covariance))
# cov = np.array([[2.41536929e+03, 2.50493520e+03, 2.58289967e+03],
#   [2.50493520e+03, 2.60502582e+03, 2.68882603e+03],
#   [2.58289967e+03, 2.68882603e+03, 2.77912300e+03]])
# cov = np.array([[4.88024409e-01, 4.87990913e-01, 4.87990913e-01],
#   [4.87990913e-01, 4.87957701e-01, 4.87957701e-01],
#   [4.87990913e-01, 4.87957701e-01, 4.87957701e-01]])
# print(np.linalg.det(cov))

# print((2*np.pi)**3 * np.linalg.det(cov))

# pixelMean = np.array([100, 200, 300]).reshape((3,1))
# pixelMeanTranspose = pixelMean.T
# product = np.dot(pixelMean, pixelMeanTranspose)
# print(pixelMean)
# print(pixelMean.shape)

# print(pixelMeanTranspose)
# print(pixelMeanTranspose.shape)
# print(product)


# diff = np.array([[ True,  True,  True],
#  [ True,  True,  True],
#  [ True,  True,  True]])

# m = np.identity(3) * np.array([2, 3, 4])
m = np.identity(3) * (np.random.rand(3) * 1000)


print(m)

In [ ]:
# K = number of gaussians to train
def trainGMM(K, pixels):
    print(f"Pixels in trainGMM: {pixels}")

    # convert pixels to int
    pixels = pixels.astype(np.float128)

    # TODO: Set convergence criteria and initialize scaling factor, gaussian mean and covariance

    # #images as array of pixels
    # print(f"len(images): {len(images)}")
    # pixels = images.reshape(-1, 3)
    # print(f"len(pixels): {len(pixels)}")
    pixels_length = pixels.shape[0]

    # we need this to be an array of k elements to compare with mean difference
    # change fill_value to change threshold
    threshold = np.full(
        shape=(K, 3), fill_value=0.00001
    )  # CHRIS # don't know if this next arg is necessary so leaving it out for now: , dtype=np.float)

    randRange = 1000

    # #starting weight is 1/K AKA scaling factor
    # weights = np.full((K), 1/K)

    # means = np.empty((K,3)) #set to empty array w/ proper shape

    # covariances = np.empty((K,3,3)) #set to empty array w/ proper shape

    # for i in range(K):

    #   #mean is avg of 3 pixels chosen at random
    #   means[i] = (pixels[np.random.randint(0,pixels_length)] +
    #               pixels[np.random.randint(0,pixels_length)] +
    #               pixels[np.random.randint(0,pixels_length)])/3

    #   #covariance matrix is identiy matrix
    #   covariances[i] = np.identity(3)

    weights = np.random.rand(K) * randRange
    weights = weights / weights.sum()
    means = np.empty((K, 3))

    covariances = np.empty((K, 3, 3))

    for i in range(K):

        pixel1 = pixels[np.random.randint(0, pixels_length)]
        pixel2 = pixels[np.random.randint(0, pixels_length)]
        pixel3 = pixels[np.random.randint(0, pixels_length)]

        # print(f"pixel1 in trainGMM mean: {pixel1}")
        print(f"pixel1.dtype: {pixel1.dtype}")
        # print(f"pixel2 in trainGMM mean: {pixel2}")
        # print(f"pixel3 in trainGMM mean: {pixel3}")

        # print(f"pixel1 + pixel2: {pixel1+pixel2}")

        # mean is avg of 3 pixels chosen at random
        # means[i] = (pixel1/3 + pixel2/3 + pixel3/3)
        means[i] = np.random.rand(3)

        # covariance matrix is identiy matrix
        covariances[i] = np.identity(3) * (np.random.rand(3) + 10**-300)

    # list of 3x3 elements
    print(f"covariance[0]: {covariances[0]}")
    print(f"means[0]: {means[0]}")
    print(f"weights: {weights}")

    maxIterations = 100

    oldMeans = means.copy() - 1

    a = np.zeros(
        (pixels_length, K)
    )  # used to store a[i][j] notice, since i iterates through all pixels its size images.size() AND j iterates over our models so its values go up to K

    iter = 0

    while not (
        iter > maxIterations or (abs(oldMeans - means) <= threshold).all()
    ):  # CHECK: not sure how this greater than operation will work

        # TODO: Main training algorithm (EM algorithm)

        print(f"shape of a {a.shape}")
        print(f"a[0][0]: {a[0][0]}")
        # expectation step GOING OFF SLIDES NOT ARTICLE (they swap i and j)

        # expectationDenominator = 0
        # #iterate over models
        # for j in range(0, K):
        # #   #iterate over ALL pixels in images
        # #   for image in images:
        # #     print("new image")
        # #     for row in image:
        # #       # print("new row")
        # #       for pixel in row:
        #         # print("new pixel")
        #         # print(pixel)
        #         # print(type(pixel))
        #         # print(means[j])

        #         # print(f"pdf output: {pdf(pixel, means[j], covariances[j])}")
        #   for pixel in pixels:
        #     expectationDenominator += weights[j] * pdf(pixel, means[j], covariances[j])

        expectationDenominator = np.zeros((len(pixels)))

        for i, pixel in enumerate(pixels):
            for j in range(0, K):
                # print(f"expectationDenominator {expectationDenominator}")
                # print(f"pdf output: {pdf(pixel, means[j], covariances[j])}")

                expectationDenominator[i] += weights[j] * pdf(
                    pixel, means[j], covariances[j]
                )

        # print(f"expectationDenominator: {expectationDenominator}")

        aLocationsSet = 0
        # iterate over models
        for j in range(0, K):
            # iterate over ALL pixels in images
            for i, pixel in enumerate(pixels):
                # compute

                # print(pixel)
                # print(f"means[j]: {means[j]}")
                # print(f"covariances[j]: {covariances[j]}")
                # print(f"weights[j]: {weights[j]}")
                # print(f"a[i][j] top: {weights[j] * pdf(pixel, means[j], covariances[j])}")
                # print(f" a[i][j] pdf: {pdf(pixel, means[j], covariances[j])}")
                # print(f"a[i][j] bottom: {expectationDenominator[i]}")

                # print("SET A[I][J]")

                aLocationsSet += 1
                a[i][j] = (
                    weights[j] * pdf(pixel, means[j], covariances[j])
                ) / expectationDenominator[i]

                # print(f"a[i][j] full: {a[i][j]}")

        print(f"a: {a}")

        # for i in range(len(a)):
        #   print(f"a[i].sum(): {a[i].sum()}")

        print(f"aLocationsSet: {aLocationsSet}")

        # maximization step

        # itrate over models
        for j in range(0, K):

            # iterate over pixels
            newMeanTop = np.zeros((1, 3))
            newMeanBottom = 0

            newVarianceTop = np.zeros((1, 3, 3))
            newVarianceBottom = 0

            newWeight = 0

            for i in range(0, pixels_length):

                # print(f"a[i][j]: {a[i][j]}")

                newMeanTop += a[i][j] * pixels[i]
                newMeanBottom += a[i][j]

                pixelMinusMean = (pixels[i] - means[j]).reshape((3, 1))

                # print(f"pixel[i]: {pixels[i]}")
                # print(f"mean[j]: {means[j]}")
                # print(f"pixelMinusMean: {pixelMinusMean}")
                # print(f"pixelMinusMeanTranspose: {pixelMinusMean.T}")
                # print(f"np.dot(pixelMinusMean, pixelMinusMean.T): {np.dot(pixelMinusMean, pixelMinusMean.T)}")

                newVarianceTop += a[i][j] * np.dot(pixelMinusMean, pixelMinusMean.T)
                newVarianceBottom += a[i][j]

                newWeight += a[i][j]

            print(f"newMeanTop: {newMeanTop}")
            print(f"newMeanBottom: {newMeanBottom}")
            print(f"newVarianceTop: {newVarianceTop}")
            print(f"newVarianceBottom: {newVarianceBottom}")
            newMean = newMeanTop / newMeanBottom
            newVariance = (newVarianceTop / newVarianceBottom) + (
                np.identity(3) * (np.random.rand(3) * 1e-6)
            )
            newWeight = newWeight / pixels_length

            oldMeans[j] = means[j]
            means[j] = newMean
            weights[j] = newWeight
            covariances[j] = newVariance

        iter += 1

        print(f"iteration number: {iter}")
        print(f"oldMeans: {oldMeans}")
        print(f"means (new): {means}")
        print(
            f"(abs(oldMeans - means) <= threshold): {(abs(oldMeans - means) <= threshold)}"
        )
        print(f"weights (new): {weights}")
        print(f"covariances (neW): {covariances}")

    return weights, means, covariances


# TESTING:
# path = "/content/train_images"
# files = "*.jpg"

# file_path = glob.glob(os.path.join(path, files))

# images = []

# print(file_path[-1:])
# for image in file_path[-1:]:
#   images.append(cv2.imread(image))


# images = np.array(images)
# weights, means, covariances = trainGMM(1, images)
# print(weights)
# print(means)
# print(covariances)


def testGMM(Model_parameters, threshold, prior):
    K, scalars, means, covariances = Model_parameters
    # TODO:  Read test images
    path = "/content/test_images"
    files = "*.jpg"
    results = "/content/results"
    os.makedirs(results, exist_ok=True)
    # Load test images
    file_path = glob.glob(os.path.join(path, files))
    pixel_values = []
    locations = []
    posteriors = []

    # TODO:  Main testing loop over all test images and use thresholding to get the orange ball
    for x in file_path:  # CHRIS
        image = cv2.imread(x)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # CHRIS
        image_pixels = []
        image_locations = []
        image_posteriors = []  # FOR QUICKER TESTING

        for i, row in enumerate(image):
            for j, pixel in enumerate(row):
                posterior = 0

                for cluster in range(K):  # Summation from article
                    posterior += scalars[cluster] * pdf(
                        pixel / np.array([179, 255, 255]),
                        means[cluster],
                        covariances[cluster],
                    )

                # print(posterior)
                # if posterior >= threshold:
                if True:
                    image_pixels.append(pixel)
                    image_locations.append((i, j))
                    image_posteriors.append(posterior)

        pixel_values.append(image_pixels)
        locations.append(image_locations)
        posteriors.append(image_posteriors)  # FOR QUICKER TESTING

    # TODO: Saving predictions to the result folder
    # outputs.append((originalImage, image))
    # results_file = os.path.join(results, os.path.basename(x))
    # cv2.imwrite(results_file,image)

    return (pixel_values, locations, posteriors)


def measureDepth(cluster_parameters):
    pass


# TODO: Identify the pixels which belong to the orange ball
# '''Enter your code here'''

# Use Cv2 operators like open and close to identiy pixels that belong
# Can use a form of normal distribution that takes the mean and standard deviation (covariance)
# Inverse sqaure curve formula

# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.optimize import curve_fit

# # Generate some sample data
# x = np.array([1, 2, 3, 4, 5])
# y = np.array([10, 2.5, 1.11, 0.625, 0.4])

# # Define the inverse square function
# def inverse_square(x, a):
#     return a / x**2

# # Fit the curve
# popt, pcov = curve_fit(inverse_square, x, y)

# # Plot the data and the fitted curve
# plt.scatter(x, y)
# plt.plot(x, inverse_square(x, *popt), 'r-')
# plt.xlabel('x')
# plt.ylabel('y')
# plt.title('Inverse Square Curve Fit')
# plt.show()


# # TODO: Use the data provided and any feature you like to obtain a model to estimate distance. eg. area of the ball on the image decreases with distance (generally follows a inverse square curve)
#   # '''Enter your code here'''

#   return distance


def plotGMM(Model_parameters, distance, pixels):
    K, scaling, mean, covariance = Model_parameters
    print(f"K in plotGMM: {K}")

    fig = plt.figure()
    axis = fig.add_subplot(111, projection="3d")

    colors = ["r", "g", "b"]

    bias = 0

    for i in range(0, K):
        print(f"scaling[{i}] in plotGMM: {scaling[i]}")
        print(f"mean[{i}] in plotGMM: {mean[i]}")
        print(f"covariance[{i}] in plotGMM: {covariance[i]}")
        u = np.linspace(0, 2 * np.pi, 100)
        v = np.linspace(0, np.pi, 100)

        x = np.outer(np.cos(u), np.sin(v))
        y = np.outer(np.sin(u), np.sin(v))
        z = np.outer(np.ones_like(u), np.cos(v))
        sphere = np.stack((x, y, z), axis=-1)[..., None]

        e, v = np.linalg.eig(covariance[i])
        s = v @ np.diag(np.sqrt(e)) @ v.T

        ellipsoid = (s @ sphere).squeeze(-1) + mean[i]

        # axis.plot_surface(*ellipsoid.transpose(2, 0, 1), rstride=4, cstride=4, color=colors[i%3], alpha=scaling[i])
        # rgba(219,101,65,255)
        axis.plot_surface(
            *ellipsoid.transpose(2, 0, 1),
            rstride=4,
            cstride=4,
            color=(219 / 255, 101 / 255, 65 / 255, scaling[i]),
        )

        # # Reshape transformed points into the correct shape for plotting
        # x_ellipsoid = ellipsoid_points[:, 0].reshape(x.shape)
        # y_ellipsoid = ellipsoid_points[:, 1].reshape(y.shape)
        # z_ellipsoid = ellipsoid_points[:, 2].reshape(z.shape)

        # # Shift by the mean of the Gaussian component
        # x_ellipsoid += mean[i][0]
        # y_ellipsoid += mean[i][1]
        # z_ellipsoid += mean[i][2]

        # # Plot the ellipsoid
        # axis.plot_wireframe(x_ellipsoid, y_ellipsoid, z_ellipsoid, color=colors[i], alpha=0.3)

    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    plt.show()
    # TODO: Visualize GMM ellipsoids
    # '''Enter your code here'''

    fig = plt.figure()
    axis = fig.add_subplot(111, projection="3d")

    pixels = pixels.transpose()

    axis.scatter(pixels[0], pixels[1], pixels[2], c=(0, 0, 0, 0.5), s=10)

    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    plt.show()

In [ ]:
scalars, means, covariances = 0, 0, 0  # Initialize GMM parameters

In [ ]:
# Main function (Algorithm 1 in the pseudocode above)


# -Write Python code to cluster the orange ball using Gaussian Mixture Model [40 points].

# -Estimate the distance to the ball [20 points].

# -Plot all the GMM ellipsoids [10 points].


# TODO: Import all python packages you need

threshold = 5e-8  # CHRIS last = 5e-12 # Threshold for clustering (Set it to an appropriate value)
K = 4  # Number of gaussians (Set it to an appropriate value)

mode_flag = 1  # (Set it to 0 for training, 1 for testing)

if mode_flag == 0:

    # TODO: Read in training images

    # Initializing file_path
    path = "/content/train_images"
    files = "*.jpg"

    file_path = glob.glob(os.path.join(path, files))

    # TODO: Iterate over training images to extract orange pixels using masks
    pixels = []

    for x in file_path:  # Looping through images
        image = cv2.imread(x)
        hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # CHRIS

        # Setting bounds for hsv mask
        lower_orange = np.array([110, 100, 100])
        upper_orange = np.array([120, 255, 255])
        mask = cv2.inRange(hsv_image, lower_orange, upper_orange)
        # Masking Images
        orange_pixels = hsv_image[mask > 0]
        pixels.extend(orange_pixels)

    pixels = np.array(pixels) / np.array([179, 255, 255])

    print(f"pixels.shape going into trainGMM: {pixels.shape}")  # CHRIS
    print(f"pixels: {pixels}")

    scalars, means, covariances = trainGMM(K, pixels)


else:
    # TODO: Load Model
    print(scalars)
    print(means)
    print(covariances)

    # print(pixels)
    # print(pixels.transpose())

    cluster_parameters = testGMM((K, scalars, means, covariances), threshold, prior)
    # distance = measureDepth(cluster_parameters)
    plotGMM((K, scalars, means, covariances), 1, pixels)

In [ ]:
# for x in file_path[-1:]: # Looping through images
#   image = cv2.imread(x)
#   print(image.shape)

# print(image.shape)
# print(image.dtype)
pixel_value_list, pixel_location_list, posteriors_list = cluster_parameters
# print(pixel_value_list[0])
# print(len(pixel_value_list))
# print(pixel_location_list[0])
# print(len(pixel_location_list))

# for image_num in range(len(pixel_value_list)):
#   for row_num in range(len(pixel_value_list[image_num])):
#     for col_num in range(len(pixel_value_list[image_num][row_num])):
#       image[image_num][pixel_location_list[row_num][col_num][0]][pixel_location_list[row_num][col_num][1]] = pixel_value_list[row_num][col_num]


threshold = 1.9e-1

for i in range(0, 8):
    image = np.zeros((640, 640, 3), dtype=np.uint8)
    fig, axes = plt.subplots(1, 1, figsize=(24, 8))

    print(i)
    for value, location, posterior in zip(
        pixel_value_list[i], pixel_location_list[i], posteriors_list[i]
    ):
        # print(location)
        if posterior >= threshold:
            image[location[0]][location[1]] = value

        # if (value > max).any:
        #   max = value
        # if (value < min).any:
        #   min = value

        # print(image.shape)
        # print(image.dtype)

        # print(max)
        # print(min)
    image = cv2.cvtColor(image, cv2.COLOR_HSV2BGR)
    # image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    axes.imshow(image)

In [ ]:
pixels, locations = cluster_parameters
print(pixels)
print(locations)

# Video Lecture
Click [here](https://www.youtube.com/watch?v=D5AcaFMY_BI) for a video lecture to help you better understand the project.

## Report
For each section of the project, explain briefly what you did, and describe any interesting problems you encountered and/or solutions you implemented. You must include the following details in your writeup:

- Your choice of color space, initialization method and number of gaussians in the GMM
- Explain why GMM is better than single gaussian
- Present your distance estimate and cluster segmentation results for each test image
- Explain strengths and limitations of your algorithm. Also, explain why the algorithm failed on some test images

As usual, your report must be full English sentences, not commented code. There is a word limit of 1500 words and no minimum length requirement.

## ***add your report here***

# Submission Guidelines

**If your submission does not comply with the following guidelines, you’ll be given ZERO credit.**

Your submission on ELMS(Canvas) must be a pdf file, following the naming convention **YourDirectoryID_proj1.pdf**. For example, xyz123_proj1.pdf.

**All your results and report should be included in this notebook. After you finished all, please export the notebook as a pdf file and submit it to ELMS(Canvas).**

# Collaboration Policy
You are encouraged to discuss the ideas with your peers. However, the code should be your own, and should be the result of you exercising your own understanding of it. If you reference anyone else’s code in writing your project, you must properly cite it in your code (in comments) and your writeup. For the full honor code refer to the CMSC426 Fall 2023 website.